In [506]:
# starting with content based filtering

In [ ]:
import numpy as np
import pandas as pd
import difflib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [508]:
# loading the data from a CSV file to pandas data
df1 = pd.read_csv("../dataset/tmdb_5000_credits.csv")
df1.head()

,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [509]:
# loading the data from a CSV file to pandas data
df2 = pd.read_csv("../dataset/tmdb_5000_movies.csv")
df2.head(1)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800


In [510]:
df = df1.merge(df2 , on="title" , how='inner')
df.head(2)

,movie_id,title,cast,crew,budget,genres,homepage,id,keywords,original_language,...,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,...,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,7.2,11800
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,...,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",6.9,4500


In [511]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   movie_id              4809 non-null   int64  
 1   title                 4809 non-null   object 
 2   cast                  4809 non-null   object 
 3   crew                  4809 non-null   object 
 4   budget                4809 non-null   int64  
 5   genres                4809 non-null   object 
 6   homepage              1713 non-null   object 
 7   id                    4809 non-null   int64  
 8   keywords              4809 non-null   object 
 9   original_language     4809 non-null   object 
 10  original_title        4809 non-null   object 
 11  overview              4806 non-null   object 
 12  popularity            4809 non-null   float64
 13  production_companies  4809 non-null   object 
 14  production_countries  4809 non-null   object 
 15  release_date         

In [512]:
selected_features = ['movie_id' , 'title','overview' , 'genres','keywords','cast','crew']
print(selected_features)
df[selected_features].isnull().sum()

['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']


movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [513]:
# Checking for Null values and filling it with empty string.
# for feature in selected_features:
#     df[feature] = df[feature].fillna('')
df.dropna(inplace=True)
df[selected_features].isnull().sum()

movie_id    0
title       0
overview    0
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [514]:
df.iloc[0][selected_features]

movie_id                                                19995
title                                                  Avatar
overview    In the 22nd century, a paraplegic Marine is di...
genres      [{"id": 28, "name": "Action"}, {"id": 12, "nam...
keywords    [{"id": 1463, "name": "culture clash"}, {"id":...
cast        [{"cast_id": 242, "character": "Jake Sully", "...
crew        [{"credit_id": "52fe48009251416c750aca23", "de...
Name: 0, dtype: object

In [515]:
import json
def preprocess(x):
    L = []
    for i in x:
        L.append(i["name"])
    return L

def preprocess_cast(x):
    L = []
    for i in x[:3]:
        L.append(i["name"])
    return L
# preprocess(json.loads(df.iloc[0]['genres']))
df['genres'] = df['genres'].apply(json.loads).apply(preprocess)
df['keywords'] = df['keywords'].apply(json.loads).apply(preprocess)
df['cast'] = df['cast'].apply(json.loads).apply(preprocess_cast)
df['crew'] = df['crew'].apply(json.loads).apply(lambda x: [i['name'] for i in x if i['job'] == 'Director'])

In [516]:
df[selected_features].iloc[0]

movie_id                                                19995
title                                                  Avatar
overview    In the 22nd century, a paraplegic Marine is di...
genres          [Action, Adventure, Fantasy, Science Fiction]
keywords    [culture clash, future, space war, space colon...
cast         [Sam Worthington, Zoe Saldana, Sigourney Weaver]
crew                                          [James Cameron]
Name: 0, dtype: object

In [517]:
df['genres'] = df['genres'].apply(lambda x: [i.replace(" ","") for i in x])
df['keywords'] = df['keywords'].apply(lambda x: [i.replace(" ","") for i in x])
df['cast'] = df['cast'].apply(lambda x: [i.replace(" ","") for i in x])
df['crew'] = df['crew'].apply(lambda x: [i.replace(" ","")  for i in x])

In [518]:
df[selected_features].iloc[0]

movie_id                                                19995
title                                                  Avatar
overview    In the 22nd century, a paraplegic Marine is di...
genres           [Action, Adventure, Fantasy, ScienceFiction]
keywords    [cultureclash, future, spacewar, spacecolony, ...
cast            [SamWorthington, ZoeSaldana, SigourneyWeaver]
crew                                           [JamesCameron]
Name: 0, dtype: object

In [519]:
# Combining all the 5 selected features
combined_string = (
    df['overview'] * 1 + " " +
    df['genres'].apply(lambda x: " ".join(x)) * 3 + " " +
    df['keywords'].apply(lambda x: " ".join(x)) * 2 + " " +
    df['cast'].apply(lambda x: " ".join(x)) * 2 + " " +
    df['crew'].apply(lambda x: " ".join(x)) * 2
)
df['tags'] = combined_string.apply(lambda x: x.lower())

In [520]:
df_new = df[['movie_id','title','tags']]
df_new.head(1)

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."


In [535]:
df_new.to_csv("../dataset_cleaned/output.csv", index=False)

In [521]:
df_new['clean_title'] = (
    df_new['title']
    .astype(str)
    .str.lower()
    .str.replace(r'[^a-z0-9]', '', regex=True)
)

C:\Users\kousi\AppData\Local\Temp\ipykernel_65688\3832098366.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_new['clean_title'] = (


In [522]:
# # Converting all the Text Data to Numbering vectors
# vectorizer = TfidfVectorizer()
# feature_vectors = vectorizer.fit_transform(df_new['clean_title'] + ' ' + df_new['tags'])

from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000 , stop_words='english')
feature_vectors = cv.fit_transform(df_new['tags'])
feature_vectors.shape

(1494, 5000)

In [523]:
# Cosine similarity of all the feature vectors
# The Thing it does is that bruteforce From Row1 to all the Rows in the DataFrame.
similarity = cosine_similarity(feature_vectors)
similarity.shape

(1494, 1494)

In [524]:
len(similarity[0])

1494

In [525]:
df_new["clean_title"].iloc[1]

'piratesofthecaribbeanatworldsend'

In [526]:
movie_names = df_new['clean_title'].tolist()
movie_names[:10]

['avatar',
 'piratesofthecaribbeanatworldsend',
 'spectre',
 'thedarkknightrises',
 'johncarter',
 'spiderman3',
 'tangled',
 'avengersageofultron',
 'harrypotterandthehalfbloodprince',
 'batmanvsupermandawnofjustice']

In [527]:
df_new.head(2)

,movie_id,title,tags,clean_title
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di...",avatar
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha...",piratesofthecaribbeanatworldsend


In [528]:
# rapidfuzz we are using to find the closest Match.
# Let us say if there are any spelling mistakes in the input array
# we need the correct spelling to find the index and check the cosine similarity
from rapidfuzz import process

movies = df_new['clean_title'].tolist()

user_input = "ironman"

find_close_match = process.extract(user_input, movies , limit=5)

print(find_close_match)

[('ironman', 100.0, 56), ('ironman3', 93.33333333333333, 27), ('ironman2', 93.33333333333333, 64), ('harleydavidsonandthemarlboroman', 75.00000000000001, 825), ('room', 67.5, 1043)]


In [529]:
df_new['clean_title'].tolist()

['avatar',
 'piratesofthecaribbeanatworldsend',
 'spectre',
 'thedarkknightrises',
 'johncarter',
 'spiderman3',
 'tangled',
 'avengersageofultron',
 'harrypotterandthehalfbloodprince',
 'batmanvsupermandawnofjustice',
 'quantumofsolace',
 'piratesofthecaribbeandeadmanschest',
 'theloneranger',
 'manofsteel',
 'theavengers',
 'piratesofthecaribbeanonstrangertides',
 'meninblack3',
 'thehobbitthebattleofthefivearmies',
 'theamazingspiderman',
 'robinhood',
 'thehobbitthedesolationofsmaug',
 'thegoldencompass',
 'titanic',
 'captainamericacivilwar',
 'jurassicworld',
 'skyfall',
 'spiderman2',
 'ironman3',
 'aliceinwonderland',
 'transformersrevengeofthefallen',
 'transformersageofextinction',
 'ozthegreatandpowerful',
 'theamazingspiderman2',
 'tronlegacy',
 'cars2',
 'greenlantern',
 'toystory3',
 'terminatorsalvation',
 'furious7',
 'worldwarz',
 'xmendaysoffuturepast',
 'jackthegiantslayer',
 'princeofpersiathesandsoftime',
 'pacificrim',
 'transformersdarkofthemoon',
 'indianajonesa

In [530]:
# Our Agenda is to first find the index of the Movie with title.
closest_match_title = find_close_match[0][0]
# closest_match_title
index_of_the_movie = df_new[df_new["clean_title"] == closest_match_title].index[0]
index_of_the_movie

np.int64(68)

In [531]:
# In the similarity_score array we have bruteforce array with all combinations
# of solutions we get list of all movies from the index.
similarity_score = list(enumerate(similarity[index_of_the_movie]))

In [532]:
# So in the below code we are basically sorting in descending order
# to find the Highest Recommended movies
sorted_similar_score = sorted(similarity_score , key= lambda x : x[1] , reverse=True)
sorted_similar_score

[(68, np.float64(0.9999999999999996)),
 (127, np.float64(0.4574447205785295)),
 (23, np.float64(0.4280933486795169)),
 (27, np.float64(0.40597241736665374)),
 (7, np.float64(0.3863958269476032)),
 (14, np.float64(0.3817148409929318)),
 (137, np.float64(0.36759581235874367)),
 (100, np.float64(0.3508724993375327)),
 (428, np.float64(0.31692699907846444)),
 (64, np.float64(0.3056598768802264)),
 (176, np.float64(0.30413390246226585)),
 (169, np.float64(0.3016162850106335)),
 (40, np.float64(0.2844912156790806)),
 (102, np.float64(0.2780406009776234)),
 (56, np.float64(0.27372051221603927)),
 (13, np.float64(0.24655905358853653)),
 (314, np.float64(0.24090158393312155)),
 (9, np.float64(0.2382552128851246)),
 (80, np.float64(0.22832791234824443)),
 (285, np.float64(0.22618371458132558)),
 (1114, np.float64(0.22618371458132558)),
 (59, np.float64(0.2208885145092328)),
 (15, np.float64(0.22000654012515564)),
 (53, np.float64(0.21900143993920138)),
 (175, np.float64(0.2176078175211597)),
 (7

In [533]:
# The below is code for top 20 Highest recommended movies
count = 0
for i in sorted_similar_score:
    if i[0] == index_of_the_movie:
        continue   # skip same movie
    
    print(df_new.iloc[i[0]]['title'])
    count += 1
    
    if count >= 10:
        break

Captain America: The First Avenger
Captain America: Civil War
Iron Man 3
Avengers: Age of Ultron
The Avengers
Ant-Man
Thor: The Dark World
Deadpool
Iron Man 2
Fantastic Four
